# Capstone Project - Rajendra Raju

**Option 1** — Built using **Python, LangChain, LangGraph, and RAG (Retrieval-Augmented Generation)**.

## What this notebook does
- Accepts PDF uploads directly in Google Colab (no repo clone needed)
- Classifies each document into one of three labels: **Cease / Uncertain / Irrelevant**
- Uses a RAG knowledge base to give the LLM legal grounding before classifying
- Routes documents through a LangGraph state machine to the right downstream agent
- Stores Cease records in SQLite, archives Irrelevant ones to JSONL
- Flags low-confidence documents for Human-in-the-Loop (HITL) review
- Writes a full audit trail for every document processed

## Classification Labels
| Label | Meaning |
|---|---|
| ✅ **Cease** | Valid cease & desist request — stored in database |
| ⚠️ **Uncertain** | Ambiguous or low-confidence — sent for human review |
| ❌ **Irrelevant** | Not a cease request — archived to flat file |


## 1) Install Dependencies

Run this cell once before anything else.
It installs:
- **LangChain / LangGraph** — workflow orchestration
- **langchain-groq** — Groq LLM integration
- **pypdf** — PDF text extraction
- **pydantic** — structured output schema
- **chromadb + sentence-transformers** — local vector store for RAG
- **langchain-community / langchain-text-splitters** — RAG document utilities


In [1]:
# Install all required packages.
# - langchain / langgraph          : workflow orchestration
# - langchain-groq                  : Groq LLM integration
# - pypdf                           : PDF text extraction
# - pydantic                        : structured LLM output schema
# - langchain-huggingface           : HuggingFace embeddings (for RAG)
# - chromadb + sentence-transformers: local in-memory vector store
# - langchain-community             : Chroma vector store wrapper
# - langchain-text-splitters        : splits knowledge into chunks
!pip -q install langchain langgraph langchain-groq pypdf pydantic typing-extensions \
    langchain-huggingface langchain-community langchain-text-splitters \
    chromadb sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


## 2) Load API Keys from Colab Secrets

Store your keys using the 🔑 icon in the left sidebar before running this cell.

| Secret | Required | Purpose |
|---|---|---|
| `GROQ_API_KEY` | ✅ Yes | Powers the LLM classifier |
| `LANGSMITH_API_KEY` | Optional | Enables LangSmith tracing |
| `ARIZE_API_KEY` | Optional | Enables Arize Phoenix observability |
| `ARIZE_SPACE_ID` | Optional | Required alongside Arize API key |


In [2]:
import os

def load_secret(name: str, required: bool = False):
    """Try Colab Secrets first, fall back to environment variables."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)  # reads from Colab's secret store
    except Exception:
        value = os.environ.get(name)  # fallback for local / CI environments

    if value:
        os.environ[name] = value   # make it available to downstream libraries
        print(f"{name}: loaded")
        return value

    if required:
        raise ValueError(f"Missing required secret: {name}")

    print(f"{name}: not provided (optional — skipping)")
    return None

# GROQ_API_KEY is mandatory — the LLM won't work without it
GROQ_API_KEY     = load_secret("GROQ_API_KEY",     required=True)
LANGSMITH_API_KEY = load_secret("LANGSMITH_API_KEY", required=False)
ARIZE_API_KEY    = load_secret("ARIZE_API_KEY",    required=False)
ARIZE_SPACE_ID   = load_secret("ARIZE_SPACE_ID",   required=False)

# Enable LangSmith tracing only when the key is present
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "Capstone Project-Rajendra"
    print("LangSmith tracing: enabled")


GROQ_API_KEY: loaded
LANGSMITH_API_KEY: loaded
ARIZE_API_KEY: loaded
ARIZE_SPACE_ID: loaded


## 3) Upload PDF Files

Run this cell to open the file picker and select one or more PDF documents.
Uploaded files are saved into the `uploaded_pdfs/` folder so the rest of
the pipeline can find them by path — no hardcoded filenames needed.

> **Outside Colab?** Drop PDF files manually into a folder called `uploaded_pdfs/`
> in the same directory as this notebook, then skip this cell.


In [3]:
import os
from pathlib import Path

# UPLOAD_DIR is also defined in Cell 8 (Imports).
# It's defined here too so this cell can run standalone if needed
# (e.g. if you want to re-upload files without re-running everything).
# Note: UPLOAD_DIR is also set in Cell 8 (Imports) — both point to the same folder.
UPLOAD_DIR = Path("uploaded_pdfs")
UPLOAD_DIR.mkdir(exist_ok=True)

try:
    from google.colab import files
    print("Select one or more PDF files to upload:")
    uploaded = files.upload()   # triggers the Colab file picker dialog

    # Save each uploaded file to UPLOAD_DIR so other cells can reference it by path
    for filename, data in uploaded.items():
        dest = UPLOAD_DIR / filename
        dest.write_bytes(data)
        print(f"  Saved: {dest}")

except ImportError:
    # Running locally — just drop PDFs into uploaded_pdfs/ manually
    print("Not running in Colab. Place PDF files in 'uploaded_pdfs/' and re-run from Cell 4.")

print("\nUpload directory:", UPLOAD_DIR.resolve())
print("Files ready:", [f.name for f in sorted(UPLOAD_DIR.glob("*.pdf"))])


 data   guides	 Ignore   README.md  'Sample Docs'


## 4) Imports and Folder Setup

This cell imports all libraries and defines the output file paths.
All results (database, archive, audit logs) are written into the `outputs/` folder
so they are easy to find and download after a run.


In [4]:
from pathlib import Path
from typing import TypedDict, Optional, Dict, Any, Literal
import json
import sqlite3
from datetime import datetime, timezone

from pypdf import PdfReader                         # extracts text from PDFs
from pydantic import BaseModel, Field               # enforces structured LLM output
from langchain_groq import ChatGroq                 # Groq-hosted LLM client
from langgraph.graph import StateGraph, END         # state machine orchestration

# ── Paths ────────────────────────────────────────────────────────────────────
# UPLOAD_DIR: where PDFs from Cell 3 are stored
UPLOAD_DIR = Path("uploaded_pdfs")
UPLOAD_DIR.mkdir(exist_ok=True)

# OUTPUT_DIR: all results go here — DB, archive, audit logs
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

DB_PATH         = OUTPUT_DIR / "cease_documents.db"      # SQLite for Cease records
ARCHIVE_PATH    = OUTPUT_DIR / "irrelevant_archive.jsonl" # flat-file for Irrelevant docs
AUDIT_PATH      = OUTPUT_DIR / "audit_log.jsonl"         # full decision trail
HITL_QUEUE_PATH = OUTPUT_DIR / "hitl_queue.jsonl"        # manual review log

print("Upload dir:", UPLOAD_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("PDFs available:", [f.name for f in sorted(UPLOAD_DIR.glob("*.pdf"))])


Data dir exists: True
Sample dir exists: True


## 5) Helper Functions

Small utility functions shared across agents:
- `utc_now_iso()` — consistent UTC timestamps for all log entries
- `append_jsonl()` — thread-safe single-line append to JSONL files
- `read_pdf_text()` — extracts and joins text from all pages of a PDF
- `get_candidate_pdfs()` — returns a deduplicated list of PDFs from the upload folder


In [5]:
def utc_now_iso() -> str:
    """Return the current UTC time as an ISO-8601 string."""
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, payload: dict) -> None:
    """Append a single JSON record to a JSONL file (one object per line)."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")


def read_pdf_text(pdf_path: Path) -> str:
    """Extract and return all text from every page of a PDF file."""
    reader = PdfReader(str(pdf_path))
    text_parts = []
    for page in reader.pages:
        # extract_text() can return None for image-only pages — default to empty string
        text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts).strip()


def get_candidate_pdfs() -> list[Path]:
    """Return a deduplicated list of PDF files from the upload directory.

    Deduplication is done by (normalised filename, file size) so the same
    document uploaded twice doesn't get processed twice.
    """
    files = sorted(UPLOAD_DIR.glob("*.pdf"))

    unique_files   = []
    seen_names     = set()
    seen_signatures = set()

    for f in files:
        normalized_name = f.name.strip().lower()
        try:
            stat = f.stat()
            signature = (normalized_name, stat.st_size)  # (name, size) pair
        except Exception:
            signature = (normalized_name, None)

        if normalized_name in seen_names or signature in seen_signatures:
            continue  # skip duplicate

        seen_names.add(normalized_name)
        seen_signatures.add(signature)
        unique_files.append(f)

    return unique_files


## 6) Database Setup

Creates a local SQLite database that stores every document classified as **Cease**.
SQLite was chosen because it requires no server, the file is portable,
and the data can be inspected in Cell 12 immediately after a run.

Columns stored per Cease record:
| Column | Description |
|---|---|
| `date_received` | UTC timestamp when the document was first loaded |
| `document_name` | Original PDF filename |
| `classification` | Always `'Cease'` for records in this table |
| `confidence` | Model confidence score (0.0 – 1.0) |
| `customer_name` | Extracted sender / customer name |
| `request_summary` | Short summary of the cease request |
| `extracted_details` | Key details pulled from the document |
| `processed_at` | UTC timestamp when the record was written |


In [6]:
# Connect to (or create) the SQLite database file
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# CREATE TABLE IF NOT EXISTS is safe to run multiple times —
# it won't wipe existing data if you re-run this cell.
cur.execute(
    '''
    CREATE TABLE IF NOT EXISTS cease_documents (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        date_received    TEXT NOT NULL,
        document_name    TEXT NOT NULL,
        classification   TEXT NOT NULL,
        confidence       REAL NOT NULL,
        customer_name    TEXT,
        request_summary  TEXT,
        extracted_details TEXT,
        processed_at     TEXT NOT NULL
    )
    '''
)
conn.commit()
print("SQLite database ready at:", DB_PATH)


SQLite ready at: outputs/cease_documents.db


## 7) RAG Knowledge Base + LLM + Structured Output Schema

### Why RAG?
Before asking the LLM to classify a document, we first retrieve relevant
legal context from a small in-memory knowledge base. This "grounding" step
significantly reduces hallucinations — the model knows what a valid Cease
request looks like *before* it reads the document.

### Knowledge Base
A ChromaDB vector store (running fully in-memory) holds a set of curated
legal rules. At classification time, the top matching rules are retrieved
and injected into the prompt alongside the document text.

### Structured Output
The LLM returns a **Pydantic object** — not free text — so every field
is typed, validated, and guaranteed to be present:

| Field | Type | Description |
|---|---|---|
| `label` | `Cease` \| `Uncertain` \| `Irrelevant` | Classification result |
| `confidence` | float (0–1) | Model's self-assessed certainty |
| `explanation` | str | Short rationale for the decision |
| `customer_name` | str | Extracted sender / customer name |
| `request_summary` | str | One-line summary of the request |
| `key_details` | str | Important details pulled from the document |

**Confidence threshold:** if the model returns a score below **0.60**,
the label is automatically overridden to `Uncertain` and routed to HITL,
regardless of what label the model chose.


In [7]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── RAG Knowledge Base ───────────────────────────────────────────────────────
# These rules teach the LLM what makes a valid Cease & Desist request
# vs. other legal documents it might encounter.
# The knowledge base runs entirely in-memory — no external service needed.
LEGAL_KNOWLEDGE = [
    "A Cease and Desist letter is a formal legal document that demands the "
    "recipient stop a specific activity such as copyright infringement, "
    "harassment, defamation, or breach of contract. It must explicitly request "
    "the cessation of communication or the disputed activity.",

    "A Letter of Authority (LoA) or Limited Power of Attorney grants a third "
    "party permission to act on behalf of the sender. This is a standard legal "
    "notice and is NOT a Cease and Desist request — classify it as Irrelevant.",

    "General business correspondence, debt resolution notices, marketing "
    "opt-out requests, and unsubscribe confirmations do not constitute Cease "
    "and Desist requests — classify these as Irrelevant.",

    "If a document contains legal language demanding all communication stop "
    "but lacks clarity, is poorly formatted, or references ambiguous activity, "
    "classify it as Uncertain and route it for human review.",
]

# Embed the knowledge using a lightweight local model (no API call needed)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Split knowledge into chunks and build the vector store
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
knowledge_docs = splitter.create_documents(LEGAL_KNOWLEDGE)
vectorstore = Chroma.from_documents(knowledge_docs, embeddings)
rag_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})  # top-2 relevant rules

print(f"RAG knowledge base ready — {len(knowledge_docs)} chunks indexed.")

# ── Pydantic Structured Output Schema ────────────────────────────────────────
class ClassificationResult(BaseModel):
    """The exact structure the LLM must return for every document."""
    label:            Literal["Cease", "Uncertain", "Irrelevant"] = Field(
                          ..., description="Final document classification")
    confidence:       float = Field(..., ge=0.0, le=1.0,
                          description="Model confidence score between 0 and 1")
    explanation:      str   = Field(...,
                          description="Short rationale for the classification decision")
    customer_name:    str   = Field(default="",
                          description="Best-effort extraction of the sender or customer name")
    request_summary:  str   = Field(default="",
                          description="One-sentence summary of the document's request")
    key_details:      str   = Field(default="",
                          description="Key facts or dates extracted from the document")

# ── LLM Initialisation ───────────────────────────────────────────────────────
# temperature=0 keeps responses deterministic — important for consistent classification
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# with_structured_output forces the LLM to return a valid ClassificationResult object
structured_llm = llm.with_structured_output(ClassificationResult)

print("LLM ready: llama-3.3-70b-versatile via Groq")


## 8) Agent Functions

Each agent is a plain Python function that takes the current workflow state,
does exactly one job, and returns the updated state. LangGraph wires them
together — agents don't call each other directly.

| Agent | Responsibility |
|---|---|
| `document_loader_agent` | Reads the PDF and extracts raw text |
| `classification_agent` | Retrieves RAG context, calls the LLM, applies confidence threshold |
| `router_agent` | Reads `final_decision` and returns the next node name |
| `database_agent` | Inserts Cease records into SQLite |
| `archive_agent` | Appends Irrelevant records to JSONL |
| `hitl_agent` | Displays document info, collects human decision, routes accordingly |
| `audit_agent` | Writes the complete decision record to the audit log |


In [8]:
# ── Workflow State ───────────────────────────────────────────────────────────
# TypedDict defines the shape of the state object passed between agents.
# total=False means every field is optional — agents only set what they produce.
class WorkflowState(TypedDict, total=False):
    pdf_path:        str    # absolute path to the PDF file
    document_name:   str    # just the filename (e.g. "cease_001.pdf")
    date_received:   str    # UTC ISO timestamp set when the document is loaded
    raw_text:        str    # full extracted PDF text
    classification:  str    # raw LLM label before confidence threshold is applied
    confidence:      float  # model confidence (0.0 – 1.0)
    explanation:     str    # model's rationale
    customer_name:   str    # extracted sender name
    request_summary: str    # one-line summary
    key_details:     str    # key facts from the document
    final_decision:  str    # the label used for routing (may differ from classification)
    reviewer_name:   str    # set by HITL agent when a human reviews
    reviewer_notes:  str    # reviewer's notes
    route:           str    # which downstream path was taken: database / archive / hitl
    processed_at:    str    # UTC ISO timestamp set when the record is written


# ── Agent 1: Document Loader ─────────────────────────────────────────────────
def document_loader_agent(state: WorkflowState) -> WorkflowState:
    """Read the PDF at state['pdf_path'] and store the extracted text in state."""
    pdf_path = Path(state["pdf_path"])
    text = read_pdf_text(pdf_path)
    return {
        **state,
        "document_name": pdf_path.name,
        "raw_text":      text or "",   # guard against None from image-only PDFs
        "date_received": utc_now_iso(),
    }


# ── RAG + LLM Classification (with retry) ────────────────────────────────────
def classify_with_retry(text: str, max_retries: int = 3) -> ClassificationResult:
    """Retrieve relevant legal context via RAG, then classify the document.

    The RAG step fetches the top-2 most relevant rules from the knowledge base
    and injects them into the prompt. This gives the model legal grounding
    specific to cease & desist processing before it reads the document.

    Retries up to max_retries times if the structured output call fails.
    """
    # Step 1 — retrieve domain context from the RAG knowledge base
    relevant_rules = rag_retriever.invoke(text[:500])  # use first 500 chars as query
    rag_context = "\n".join(doc.page_content for doc in relevant_rules)

    # Step 2 — build the classification prompt with RAG context injected
    prompt = f"""
You are a legal document classifier for a Cease & Desist processing workflow.

LEGAL CONTEXT (use this to guide your decision):
{rag_context}

CLASSIFICATION RULES:
- Cease      : Document clearly and explicitly demands cessation of communication or a specific activity.
- Irrelevant : Document is unrelated to a cease request (e.g. LoA, general notice, marketing opt-out).
- Uncertain  : Content is ambiguous, incomplete, or weakly worded — or if you are unsure.

CONFIDENCE RULE:
- If your confidence is below 0.60, still return your best label and confidence.
  Downstream logic will automatically convert low-confidence results to Uncertain.

EXTRACTION:
- Extract customer_name, request_summary, and key_details from the document.
- If a field cannot be determined, leave it as an empty string.

DOCUMENT TEXT:
{text[:12000]}
"""

    # Step 3 — call the LLM with retry logic
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return structured_llm.invoke(prompt)
        except Exception as e:
            last_error = e
            print(f"  [Classifier] Retry {attempt}/{max_retries}: {e}")
    raise RuntimeError(f"Classifier failed after {max_retries} retries: {last_error}")


# ── Agent 2: Classification ───────────────────────────────────────────────────
def classification_agent(state: WorkflowState) -> WorkflowState:
    """Run RAG-augmented classification and apply the confidence threshold.

    If the model's confidence is below 0.60, final_decision is set to
    'Uncertain' so the router sends it to HITL — even if the model picked
    Cease or Irrelevant.
    """
    text = state.get("raw_text", "").strip()

    # Handle image-only or empty PDFs gracefully
    if not text:
        return {
            **state,
            "classification": "Uncertain",
            "confidence":     0.0,
            "explanation":    "PDF text extraction returned empty content — possible image-only PDF.",
            "customer_name":  "",
            "request_summary": "",
            "key_details":    "",
            "final_decision": "Uncertain",
        }

    try:
        result = classify_with_retry(text)

        # Apply confidence threshold: low-confidence results go to HITL
        final_label = result.label if float(result.confidence) >= 0.60 else "Uncertain"

        return {
            **state,
            "classification":  result.label,           # raw model label
            "confidence":      float(result.confidence),
            "explanation":     result.explanation,
            "customer_name":   result.customer_name,
            "request_summary": result.request_summary,
            "key_details":     result.key_details,
            "final_decision":  final_label,            # used by router
        }
    except Exception as e:
        # If classification fails completely, route to HITL — never drop a document
        return {
            **state,
            "classification":  "Uncertain",
            "confidence":      0.0,
            "explanation":     f"Classification error — routed for manual review: {e}",
            "customer_name":   "",
            "request_summary": "",
            "key_details":     "",
            "final_decision":  "Uncertain",
        }


# ── Router (conditional edge function) ───────────────────────────────────────
def router_agent(state: WorkflowState) -> str:
    """Return the name of the next graph node based on final_decision."""
    decision = state.get("final_decision")

    # Defensive fallback — should not normally be needed
    if not decision:
        decision = state.get("classification", "Uncertain")
        if state.get("confidence", 0.0) < 0.60:
            decision = "Uncertain"

    if decision == "Cease":      return "database"
    if decision == "Irrelevant": return "archive"
    return "hitl"


# ── Agent 3: Database (Cease documents) ──────────────────────────────────────
def database_agent(state: WorkflowState) -> WorkflowState:
    """Insert a Cease document record into the SQLite database."""
    processed_at = utc_now_iso()
    try:
        cur.execute(
            """
            INSERT INTO cease_documents
            (date_received, document_name, classification, confidence,
             customer_name, request_summary, extracted_details, processed_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                state["date_received"],
                state["document_name"],
                "Cease",
                state.get("confidence", 0.0),
                state.get("customer_name",   ""),
                state.get("request_summary", ""),
                state.get("key_details",     ""),
                processed_at,
            )
        )
        conn.commit()  # persist immediately so data isn't lost if a later step fails
    except Exception as e:
        conn.rollback()  # undo partial write on failure
        raise RuntimeError(f"Database write failed for '{state.get('document_name')}': {e}") from e
    return {**state, "final_decision": "Cease", "route": "database", "processed_at": processed_at}


# ── Agent 4: Archive (Irrelevant documents) ───────────────────────────────────
def archive_agent(state: WorkflowState) -> WorkflowState:
    """Append an Irrelevant document record to the JSONL archive file."""
    processed_at = utc_now_iso()
    append_jsonl(ARCHIVE_PATH, {
        "date_received": state["date_received"],
        "document_name": state["document_name"],
        "classification": "Irrelevant",
        "confidence":     state.get("confidence", 0.0),
        "processed_at":   processed_at,
    })
    return {**state, "final_decision": "Irrelevant", "route": "archive", "processed_at": processed_at}


# ── HITL preset map (optional) ────────────────────────────────────────────────
# Pre-fill decisions for specific filenames to make batch runs non-interactive.
# Example:
#   PRESET_HITL_REVIEW = {
#       "suspicious_doc.pdf": {
#           "decision": "approved",       # 'approved' -> Cease, 'rejected' -> Irrelevant
#           "reviewer_name": "Rajendra",
#           "reviewer_notes": "Confirmed cease request after manual read."
#       }
#   }
PRESET_HITL_REVIEW = {}

# NOTE: INTERACTIVE_HITL is intentionally configured in Cell 10 (Test Configuration)
# so it stays together with TEST_MODE. Do not set it here.


def normalize_human_decision(value: str) -> str:
    """Map human input variants to canonical Cease / Irrelevant labels."""
    value = (value or "").strip().lower()
    if value in {"approved", "approve", "a", "cease"}:
        return "Cease"
    if value in {"rejected", "reject", "r", "irrelevant"}:
        return "Irrelevant"
    return "Irrelevant"  # safe default


# ── Agent 5: Human-in-the-Loop (HITL) ────────────────────────────────────────
def hitl_agent(state: WorkflowState) -> WorkflowState:
    """Present Uncertain documents for human review and route the final decision.

    Priority order:
    1. Preset map  — used when a filename is listed in PRESET_HITL_REVIEW
    2. Interactive — prompts the user via input() when INTERACTIVE_HITL is True
    3. Fallback    — defaults to Irrelevant for fully automated batch runs
    """
    file_name = state["document_name"]

    if file_name in PRESET_HITL_REVIEW:
        # Use pre-configured decision (useful for reproducible test runs)
        review         = PRESET_HITL_REVIEW[file_name]
        decision       = normalize_human_decision(review.get("decision", "rejected"))
        reviewer_name  = review.get("reviewer_name", "Rajendra")
        reviewer_notes = review.get("reviewer_notes", "Preset manual review")

    elif INTERACTIVE_HITL:
        # Show the model's reasoning so the reviewer can make an informed decision
        print("\n" + "=" * 50)
        print("  MANUAL REVIEW REQUIRED")
        print("=" * 50)
        print(f"  Document   : {file_name}")
        print(f"  Model label: {state.get('classification')}")
        print(f"  Confidence : {state.get('confidence')}")
        print(f"  Explanation: {state.get('explanation')}")
        print(f"  Summary    : {state.get('request_summary')}")
        print("=" * 50)
        print("  approved -> treat as Cease  |  rejected -> treat as Irrelevant")

        human_decision = input("\n  Enter decision (approved / rejected): ").strip()
        decision       = normalize_human_decision(human_decision)
        reviewer_name  = input("  Reviewer name: ").strip() or "Rajendra"
        reviewer_notes = input("  Reviewer notes: ").strip() or "Manual review completed."

    else:
        # Non-interactive fallback for batch execution
        decision       = "Irrelevant"
        reviewer_name  = "Rajendra"
        reviewer_notes = "Non-interactive fallback — defaulted to Irrelevant."

    # Apply the human decision through the appropriate storage agent
    # Both database_agent and archive_agent set the 'route' field in state
    updated = {**state, "final_decision": decision,
               "reviewer_name": reviewer_name, "reviewer_notes": reviewer_notes}
    updated = database_agent(updated) if decision == "Cease" else archive_agent(updated)

    # Record the human review decision separately for audit and compliance
    append_jsonl(HITL_QUEUE_PATH, {
        "document_name":  file_name,
        "model_label":    state.get("classification"),
        "human_decision": "approved" if decision == "Cease" else "rejected",
        "final_decision": decision,
        "reviewer_name":  reviewer_name,
        "reviewer_notes": reviewer_notes,
        "processed_at":   updated.get("processed_at", utc_now_iso()),
    })

    return updated


# ── Agent 6: Audit ────────────────────────────────────────────────────────────
def audit_agent(state: WorkflowState) -> WorkflowState:
    """Write the complete decision record to the audit log.

    Every document — regardless of classification — gets an audit entry.
    This is the compliance trail that can be reviewed after any run.
    """
    append_jsonl(AUDIT_PATH, {
        "date_received":  state.get("date_received"),
        "document_name":  state.get("document_name"),
        "classification": state.get("classification"),   # raw model label
        "confidence":     state.get("confidence"),
        "explanation":    state.get("explanation"),
        "final_decision": state.get("final_decision"),   # actual routing decision
        "route":          state.get("route"),             # database / archive / hitl
        "reviewer_name":  state.get("reviewer_name", ""),
        "reviewer_notes": state.get("reviewer_notes", ""),
        "processed_at":   state.get("processed_at", utc_now_iso()),
    })
    return state  # audit agent only logs — it does not modify state


print("All agent functions defined.")


## 9) Build the LangGraph Workflow

LangGraph compiles the agents into a directed state machine.
Each node is an agent function, and edges define which node runs next.
The conditional edge after `classifier` is what enables smart routing —
the `router_agent` function reads `final_decision` and returns the
correct next node name.

```
document_loader → classifier → [database | archive | hitl] → audit → END
```


In [9]:
graph = StateGraph(WorkflowState)

# Register all agent functions as named nodes in the graph
graph.add_node("document_loader", document_loader_agent)
graph.add_node("classifier",      classification_agent)
graph.add_node("database",        database_agent)
graph.add_node("archive",         archive_agent)
graph.add_node("hitl",            hitl_agent)
graph.add_node("audit",           audit_agent)

# Entry point — every document starts here
graph.set_entry_point("document_loader")

# Fixed edge: loader always flows into classifier
graph.add_edge("document_loader", "classifier")

# Conditional edge: router_agent decides which storage node to use
# based on the final_decision field in state
graph.add_conditional_edges(
    "classifier",
    router_agent,
    {
        "database": "database",   # Cease → SQLite
        "archive":  "archive",    # Irrelevant → JSONL
        "hitl":     "hitl",       # Uncertain → human review
    }
)

# All three storage paths converge at the audit node before ending
graph.add_edge("database", "audit")
graph.add_edge("archive",  "audit")
graph.add_edge("hitl",     "audit")
graph.add_edge("audit",    END)

# Compile — this validates the graph structure and creates the runnable app
app = graph.compile()
print("LangGraph workflow compiled successfully.")


LangGraph workflow compiled successfully.


## 10) Test Configuration

Choose how many PDFs to process in this run by setting `TEST_MODE`:

| Value | Behaviour |
|---|---|
| `"one"` | Process the first uploaded PDF only — good for quick testing |
| `"two"` | Process the first two PDFs |
| `"all"` | Process every uploaded PDF |

Interactive HITL (manual review prompts) is enabled for all modes.
To disable it for a fully automated batch run, add entries to `PRESET_HITL_REVIEW`
in Cell 8 and set `INTERACTIVE_HITL = False`.


In [11]:
TEST_MODE = "all"   # options: "one" | "two" | "all"

# Interactive HITL: enabled for single/pair testing, disabled for full batch runs.
# Change to True if you want manual prompts even during an 'all' run.
INTERACTIVE_HITL = TEST_MODE in {"one", "two"}

# Collect all uploaded PDFs and apply test mode limit
all_pdfs = get_candidate_pdfs()

if TEST_MODE == "one":
    selected_pdfs = all_pdfs[:1]
elif TEST_MODE == "two":
    selected_pdfs = all_pdfs[:2]
else:
    selected_pdfs = all_pdfs  # process everything

# Guard: warn the user if no PDFs were found
if not selected_pdfs:
    print("⚠️  No PDF files found in 'uploaded_pdfs/'.")
    print("   Please run Cell 3 (Upload PDF Files) first, then re-run this cell.")
else:
    print(f"Test mode    : {TEST_MODE}")
    print(f"PDFs selected: {len(selected_pdfs)} of {len(all_pdfs)}")
    print(f"Interactive HITL: {INTERACTIVE_HITL}")
    print("\nFiles to process:")
    for item in selected_pdfs:
        print(" -", item.name)


Selected PDFs:
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA2.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA3.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA4.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA5.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA6.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA7.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA8.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LOA9.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/LoA1.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_1.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_2.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_3.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_4.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/bw_doc_5.pdf
- Rajendra-Training/day5/capstone-project/data/pdfs/notice_1.pdf
- Rajendra-Training/day5/capstone-project/data

## 11) Execute the Pipeline

This cell runs the LangGraph workflow on each selected PDF.
Each document is processed independently — if one fails, the error is
logged to `outputs/processing_errors.jsonl` and the loop continues.


In [13]:
results = []

for pdf_path in selected_pdfs:
    print(f"\nProcessing: {pdf_path.name}")
    print("-" * 40)
    try:
        # Each document starts with just its file path in the state.
        # The graph fills in all other fields as it runs through each agent.
        initial_state: WorkflowState = {"pdf_path": str(pdf_path)}
        final_state = app.invoke(initial_state)
        results.append(final_state)

        # Print a summary after each document is processed
        print(f"  Document   : {final_state.get('document_name')}")
        print(f"  Model label: {final_state.get('classification')}  "
              f"(confidence: {final_state.get('confidence', 0):.2f})")
        print(f"  Final decision: {final_state.get('final_decision')}")
        print(f"  Route      : {final_state.get('route')}")

    except Exception as e:
        # Log the error without stopping the loop — other PDFs still get processed
        error_record = {
            "document_name": pdf_path.name,
            "error":         str(e),
            "processed_at":  utc_now_iso()
        }
        append_jsonl(OUTPUT_DIR / "processing_errors.jsonl", error_record)
        print(f"  ERROR: {e} — logged to processing_errors.jsonl")

print(f"\nDone. Processed {len(results)} of {len(selected_pdfs)} documents successfully.")



Processed: LOA2.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA3.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA4.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA5.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA6.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

Processed: LOA7.pdf
Model label: Irrelevant
Confidence: 0.8
Final decision: Irrelevant
Route: archive

Processed: LOA8.pdf
Model label: Cease
Confidence: 0.8
Final decision: Cease
Route: database

Processed: LOA9.pdf
Model label: Cease
Confidence: 0.85
Final decision: Cease
Route: database

Processed: LoA1.pdf
Model label: Cease
Confidence: 0.95
Final decision: Cease
Route: database

--- Manual Review Required ---
Document: bw_doc_1.pdf
Model label: Uncertain
Confidence: 0.0
Explanation: PDF text extraction returned em

## 12) Review Outputs

Inspect the results after a run.
All output files are in the `outputs/` folder — you can download them
from the Colab file browser (left sidebar → folder icon).

| File | What it contains |
|---|---|
| `cease_documents.db` | SQLite records for all Cease documents |
| `irrelevant_archive.jsonl` | Flat-file records for Irrelevant documents |
| `audit_log.jsonl` | Full decision trail — every document processed |
| `hitl_queue.jsonl` | Human review decisions and reviewer notes |
| `processing_errors.jsonl` | Any per-file errors (only exists if errors occurred) |


In [14]:
# ── SQLite: show all stored Cease records ────────────────────────────────────
print("Output file locations:")
print("  DB     :", DB_PATH)
print("  Archive:", ARCHIVE_PATH)
print("  Audit  :", AUDIT_PATH)
print("  HITL   :", HITL_QUEUE_PATH)

rows = cur.execute(
    "SELECT date_received, document_name, customer_name, request_summary, extracted_details "
    "FROM cease_documents"
).fetchall()

print(f"\nCease documents in SQLite ({len(rows)} records):")
for row in rows:
    print(" ", row)


DB file: outputs/cease_documents.db
Archive file: outputs/irrelevant_archive.jsonl
Audit file: outputs/audit_log.jsonl
HITL file: outputs/hitl_queue.jsonl

Cease documents stored in SQLite:
('2026-03-25T19:21:36.421210+00:00', 'LOA2.pdf', 'LOVETTA CANNUNZIATA', 'Cease and desist all communications regarding the account', 'Limited Power of Attorney granted to LAW OFFICES OF DONALD A. GREEN, APLC')
('2026-03-25T19:21:36.992797+00:00', 'LOA3.pdf', 'BELLINA DAVANZO AMOEDO, RADCLIFF', 'Request to cease and desist all communications regarding the account and grant a Limited Power of Attorney to Five Lakes Law Group PLLC', 'Account/Reference#: _________ _, Limited Power of Attorney granted to Five Lakes Law Group PLLC')
('2026-03-25T19:21:37.942048+00:00', 'LOA4.pdf', 'DAVANZO,BELLINA AMOEDO,RADCLIFF', 'Cease and desist all communications regarding the account', 'Limited Power of Attorney granted to Five Lakes Law Group PLLC')
('2026-03-25T19:21:38.711011+00:00', 'LOA5.pdf', 'RANDLE L WIDHALM

In [15]:
def preview_jsonl(path: Path, limit: int = 5):
    """Print up to `limit` records from a JSONL file."""
    print(f"\n── {path.name} {'─' * max(0, 40 - len(path.name))}")
    if not path.exists():
        print("  (file not yet created — no records of this type)")
        return
    with open(path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if idx >= limit:
                print(f"  ... (showing first {limit} records)")
                break
            print(" ", json.loads(line))

preview_jsonl(ARCHIVE_PATH)
preview_jsonl(AUDIT_PATH)
preview_jsonl(HITL_QUEUE_PATH)

# ── Pipeline summary ──────────────────────────────────────────────────────────
# Count outcomes across all processed documents
if results:
    cease_count      = sum(1 for r in results if r.get("final_decision") == "Cease")
    irrelevant_count = sum(1 for r in results if r.get("final_decision") == "Irrelevant")
    uncertain_count  = sum(1 for r in results if r.get("final_decision") == "Uncertain")
    hitl_count       = sum(1 for r in results if r.get("route") == "hitl" or
                           (r.get("reviewer_name") and r.get("reviewer_name") != ""))

    print("\n" + "=" * 50)
    print("  PIPELINE SUMMARY")
    print("=" * 50)
    print(f"  Total processed : {len(results)}")
    print(f"  ✅ Cease        : {cease_count}")
    print(f"  ❌ Irrelevant   : {irrelevant_count}")
    print(f"  ⚠️  Uncertain    : {uncertain_count}")
    print(f"  👤 HITL reviews : {hitl_count}")
    print("=" * 50)

# Close the SQLite connection cleanly at the end of the session
conn.close()
print("\nSQLite connection closed.")
print("All output files saved in:", OUTPUT_DIR.resolve())



Preview -> outputs/irrelevant_archive.jsonl
{'date_received': '2026-03-25T19:21:41.973317+00:00', 'document_name': 'LOA7.pdf', 'classification': 'Irrelevant', 'confidence': 0.8, 'processed_at': '2026-03-25T19:21:42.649640+00:00'}
{'date_received': '2026-03-25T19:21:48.271836+00:00', 'document_name': 'bw_doc_1.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:23:42.224237+00:00'}
{'date_received': '2026-03-25T19:23:42.231513+00:00', 'document_name': 'bw_doc_2.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:00.941266+00:00'}
{'date_received': '2026-03-25T19:24:00.948039+00:00', 'document_name': 'bw_doc_3.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:25.266776+00:00'}
{'date_received': '2026-03-25T19:24:25.274469+00:00', 'document_name': 'bw_doc_4.pdf', 'classification': 'Irrelevant', 'confidence': 0.0, 'processed_at': '2026-03-25T19:24:33.961103+00:00'}

Preview 

## 13) What This System Covers

### Capstone Requirements
| Requirement | Implementation |
|---|---|
| Multiple Agents | 6 specialised agents — loader, classifier, database, archive, HITL, audit |
| RAG Knowledge Base | ChromaDB vector store with legal domain rules injected into every prompt |
| Human-in-the-Loop | Interactive prompts + preset map + non-interactive fallback |
| Database Interaction | SQLite stores all Cease records with metadata |
| Flat-File Archiving | JSONL archive for Irrelevant documents |
| Audit Trail | Every decision logged to `audit_log.jsonl` |

### Engineering Decisions
- **Confidence threshold (< 0.60 → Uncertain)** — prevents borderline cases from
  being auto-classified and ensures they always get a human review
- **3 retries on structured output** — handles transient LLM API errors gracefully
  without stopping the pipeline
- **Per-file error isolation** — a failure on one PDF is logged and skipped;
  the remaining documents are still processed
- **RAG context injection** — each classification call is grounded in curated
  legal rules, improving accuracy on edge cases like LoAs and general notices
- **File-based uploads** — no repo clone required; works with any PDF the user uploads
